In [1]:
# ─── STEP 1: Import Libraries ─────────────────────────────────────────────────
import random
import pandas as pd
from textblob import TextBlob
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")   # save to file instead of showing a window
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ══════════════════════════════════════════════════════════════
# STEP 1 — LOAD THE CSV DATASET
# ══════════════════════════════════════════════════════════════
df = pd.read_csv(r"C:\Users\mayan\OneDrive\Desktop\Module 4 codes\ca\customer_reviews.csv")

print(f"      Rows loaded   : {len(df):,}")
print(f"      Columns       : {list(df.columns)}")
print(f"\n      First 3 rows preview:")
print(df.head(3).to_string(index=False))

      Rows loaded   : 40,000
      Columns       : ['review_id', 'review', 'category', 'platform', 'rating', 'date', 'month', 'year', 'sentiment']

      First 3 rows preview:
 review_id                                               review         category platform  rating       date month  year sentiment
         1 Product smells bad and looks nothing like described.     Toys & Games    Nykaa       2 13-10-2023   Oct  2023  Negative
         2                 Stopped working after a week. Awful.         Clothing   Myntra       1 03-08-2023   Aug  2023  Negative
         3                       Average quality for the price. Sports & Fitness   Myntra       3 29-02-2024   Feb  2024   Neutral


In [3]:
# ══════════════════════════════════════════════════════════════
# STEP 2 — CLEAN THE TEXT
# ══════════════════════════════════════════════════════════════
print("\n[2/5] Cleaning review text ...")

def clean_text(text):
    return str(text).lower().strip()

df["clean_review"] = df["review"].apply(clean_text)
print("      Text cleaning complete.")



[2/5] Cleaning review text ...
      Text cleaning complete.


In [4]:
# ══════════════════════════════════════════════════════════════
# STEP 3 — SENTIMENT ANALYSIS ENGINE  (Component A)
# ══════════════════════════════════════════════════════════════
print("\n[3/5] Running TextBlob sentiment analysis on 40,000 reviews ...")
print("      (This may take 20-40 seconds...)")

def analyse_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        label = "Positive"
    elif polarity < -0.1:
        label = "Negative"
    else:
        label = "Neutral"
    return label, round(polarity, 3)

df[["pred_sentiment", "score"]] = df["clean_review"].apply(
    lambda t: pd.Series(analyse_sentiment(t))
)

print("      Done! Sample predictions:")
print(df[["review", "pred_sentiment", "score"]].head(6).to_string(index=False))


[3/5] Running TextBlob sentiment analysis on 40,000 reviews ...
      (This may take 20-40 seconds...)
      Done! Sample predictions:
                                              review pred_sentiment  score
Product smells bad and looks nothing like described.       Negative  -0.70
                Stopped working after a week. Awful.       Negative  -1.00
                      Average quality for the price.       Negative  -0.15
         False advertising. Nothing like the photos.       Negative  -0.40
              Wonderful product. My family loves it.       Positive   1.00
               Never again. The worst purchase ever.       Negative  -1.00


In [5]:
# ══════════════════════════════════════════════════════════════
# STEP 4 — INSIGHT GENERATOR  (Component C)
# ══════════════════════════════════════════════════════════════
print("\n[4/5] Generating business insights ...")

def generate_insights(data):
    total  = len(data)
    counts = data["pred_sentiment"].value_counts()

    pos_pct = round(counts.get("Positive", 0) / total * 100, 1)
    neg_pct = round(counts.get("Negative", 0) / total * 100, 1)
    neu_pct = round(counts.get("Neutral",  0) / total * 100, 1)
    avg_score = round(data["score"].mean(), 3)

    # Category with most negative reviews
    neg_by_cat = (
        data[data["pred_sentiment"] == "Negative"]
        .groupby("category").size()
        .sort_values(ascending=False)
    )
    worst_cat = neg_by_cat.index[0] if not neg_by_cat.empty else "N/A"

    # Platform with most negative reviews
    neg_by_plat = (
        data[data["pred_sentiment"] == "Negative"]
        .groupby("platform").size()
        .sort_values(ascending=False)
    )
    worst_platform = neg_by_plat.index[0] if not neg_by_plat.empty else "N/A"

    # Best month by average score
    month_order = ["Jan","Feb","Mar","Apr","May","Jun",
                   "Jul","Aug","Sep","Oct","Nov","Dec"]
    month_avg = (
        data.groupby("month")["score"].mean()
        .reindex(month_order).dropna()
    )
    best_month = month_avg.idxmax() if not month_avg.empty else "N/A"

    # Top 5 unique negative review texts
    top_complaints = (
        data[data["pred_sentiment"] == "Negative"]["review"]
        .unique()[:5].tolist()
    )

    # Yearly trend
    yearly_avg = (
        data.groupby("year")["score"].mean()
        .sort_index().round(3)
    )

    return {
        "total":          total,
        "pos_pct":        pos_pct,
        "neg_pct":        neg_pct,
        "neu_pct":        neu_pct,
        "avg_score":      avg_score,
        "worst_cat":      worst_cat,
        "worst_platform": worst_platform,
        "best_month":     best_month,
        "top_complaints": top_complaints,
        "yearly_avg":     yearly_avg,
        "neg_by_cat":     neg_by_cat,
        "month_avg":      month_avg,
    }

insights = generate_insights(df)
print("      Insights generated.")



[4/5] Generating business insights ...
      Insights generated.


In [6]:
# ══════════════════════════════════════════════════════════════
# STEP 5 — VISUALIZATION LAYER  (Component D)
# ══════════════════════════════════════════════════════════════
print("\n[5/5] Creating and saving charts ...")

# Chart 1 — Pie: Sentiment Distribution
fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(
    [insights["pos_pct"], insights["neg_pct"], insights["neu_pct"]],
    labels=["Positive", "Negative", "Neutral"],
    colors=["#4CAF50", "#F44336", "#FFC107"],
    autopct="%1.1f%%", startangle=140,
    textprops={"fontsize": 12}
)
ax.set_title("Customer Sentiment Distribution (40,000 Reviews)",
             fontsize=13, fontweight="bold")
fig.savefig("chart1_sentiment_pie.png", bbox_inches="tight")
plt.close(fig)
print("      Saved: chart1_sentiment_pie.png")

# Chart 2 — Bar: Sentiment by Category
cat_data = df.groupby(["category", "pred_sentiment"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(12, 5))
cat_data.plot(kind="bar", ax=ax,
              color={"Negative":"#F44336","Neutral":"#FFC107","Positive":"#4CAF50"},
              edgecolor="white")
ax.set_title("Sentiment Count by Product Category", fontsize=13, fontweight="bold")
ax.set_xlabel("Category")
ax.set_ylabel("Number of Reviews")
ax.legend(title="Sentiment")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig("chart2_category_bar.png", bbox_inches="tight")
plt.close(fig)
print("      Saved: chart2_category_bar.png")

# Chart 3 — Line: Monthly Sentiment Trend
month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
monthly = df.groupby("month")["score"].mean().reindex(month_order).dropna()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values, marker="o",
        color="#2196F3", linewidth=2.2, markersize=7)
ax.axhline(0, color="grey", linestyle="--", linewidth=0.9)
ax.fill_between(monthly.index, monthly.values, 0,
                where=(monthly.values >= 0), alpha=0.18, color="#4CAF50")
ax.fill_between(monthly.index, monthly.values, 0,
                where=(monthly.values <  0), alpha=0.18, color="#F44336")
ax.set_title("Average Sentiment Score — Monthly Trend", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Avg Polarity Score (-1 to +1)")
fig.tight_layout()
fig.savefig("chart3_monthly_trend.png", bbox_inches="tight")
plt.close(fig)
print("      Saved: chart3_monthly_trend.png")

# Chart 4 — Bar: Yearly Average Sentiment
yearly = insights["yearly_avg"]
colors_yr = ["#2196F3","#FF9800","#9C27B0","#4CAF50"][:len(yearly)]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(yearly.index.astype(str), yearly.values,
              color=colors_yr, edgecolor="white", width=0.5)
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
for bar, val in zip(bars, yearly.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            str(val), ha="center", va="bottom",
            fontsize=11, fontweight="bold")
ax.set_title("Yearly Average Sentiment Score", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Avg Polarity Score")
fig.tight_layout()
fig.savefig("chart4_yearly_trend.png", bbox_inches="tight")
plt.close(fig)
print("      Saved: chart4_yearly_trend.png")



[5/5] Creating and saving charts ...
      Saved: chart1_sentiment_pie.png
      Saved: chart2_category_bar.png
      Saved: chart3_monthly_trend.png
      Saved: chart4_yearly_trend.png


In [7]:
# ══════════════════════════════════════════════════════════════
# COMPONENT B — RULE-BASED BI CHATBOT
# ══════════════════════════════════════════════════════════════

def chatbot(question, ins):
  
    q = question.lower().strip()

    if "overall" in q and "sentiment" in q:
        return (
            f"Overall sentiment across {ins['total']:,} reviews:\n"
            f"  Positive : {ins['pos_pct']}%\n"
            f"  Negative : {ins['neg_pct']}%\n"
            f"  Neutral  : {ins['neu_pct']}%\n"
            f"  Avg polarity score: {ins['avg_score']} (scale: -1 to +1)"
        )

    elif any(w in q for w in ["complaint", "issue", "problem", "negative review"]):
        lines = "\n  - ".join(ins["top_complaints"])
        return f"Top 5 customer complaints:\n  - {lines}"

    elif any(w in q for w in ["trend", "time", "month", "change over"]):
        return (
            f"Best month for customer sentiment: {ins['best_month']}.\n"
            "See chart3_monthly_trend.png and chart4_yearly_trend.png."
        )

    elif "platform" in q:
        return (
            f"Platform with the most negative reviews: {ins['worst_platform']}.\n"
            "Consider reviewing seller onboarding standards on this platform."
        )

    elif any(w in q for w in ["worst", "bad category", "poor", "category"]):
        return (
            f"Category with the most negative reviews: {ins['worst_cat']}.\n"
            "Recommendation: Focus quality improvements here first."
        )

    elif any(w in q for w in ["recommend", "suggest", "improve", "advice"]):
        return (
            "Business Recommendations:\n"
            f"  1. Improve quality control in '{ins['worst_cat']}' category.\n"
            f"  2. Audit seller performance on {ins['worst_platform']}.\n"
            "  3. Send post-purchase emails to boost positive review volume.\n"
            "  4. Use positive review themes in marketing campaigns.\n"
            "  5. Publicly respond to all negative reviews.\n"
            "  6. Track monthly sentiment score as a live KPI."
        )

    elif any(w in q for w in ["total", "how many", "count", "records"]):
        return f"Total reviews analysed: {ins['total']:,}"

    elif any(w in q for w in ["score", "average", "polarity"]):
        return (
            f"Average sentiment polarity: {ins['avg_score']}\n"
            "(Scale: -1.0 = very negative, 0 = neutral, +1.0 = very positive)"
        )

    elif any(w in q for w in ["satisfied", "satisfaction", "happy"]):
        satisfied = int(ins["total"] * ins["pos_pct"] / 100)
        return (
            f"{ins['pos_pct']}% of customers left positive reviews.\n"
            f"That equals ~{satisfied:,} out of {ins['total']:,} customers."
        )

    elif any(w in q for w in ["help", "what can", "options"]):
        return (
            "Questions I can answer:\n"
            "  1. What is overall customer sentiment?\n"
            "  2. What are the top complaints?\n"
            "  3. How has sentiment changed over time?\n"
            "  4. Which category has the worst reviews?\n"
            "  5. Which platform has the most negative reviews?\n"
            "  6. What are your recommendations?\n"
            "  7. How many total reviews were analysed?\n"
            "  8. What is the average sentiment score?\n"
            "  9. How many customers are satisfied?\n"
            "  Type 'exit' to quit."
        )

    else:
        return "I didn't understand that. Type 'help' to see available questions."


# ── Chatbot demo run ──────────────────────────────────────────
DEMO_QUESTIONS = [
    "What is overall customer sentiment?",
    "What are the top complaints?",
    "How has sentiment changed over time?",
    "Which category has the worst reviews?",
    "Which platform has the most negative reviews?",
    "How many customers are satisfied?",
    "What are your recommendations?",
]

print("\n" + "─" * 62)
print("  CHATBOT DEMO")
print("─" * 62)

for q in DEMO_QUESTIONS:
    print(f"\nYou : {q}")
    print(f"Bot : {chatbot(q, insights)}")

print("\n" + "─" * 62)



──────────────────────────────────────────────────────────────
  CHATBOT DEMO
──────────────────────────────────────────────────────────────

You : What is overall customer sentiment?
Bot : Overall sentiment across 40,000 reviews:
  Positive : 56.5%
  Negative : 25.3%
  Neutral  : 18.2%
  Avg polarity score: 0.192 (scale: -1 to +1)

You : What are the top complaints?
Bot : Top 5 customer complaints:
  - Product smells bad and looks nothing like described.
  - Stopped working after a week. Awful.
  - Average quality for the price.
  - False advertising. Nothing like the photos.
  - Never again. The worst purchase ever.

You : How has sentiment changed over time?
Bot : Best month for customer sentiment: Jul.
See chart3_monthly_trend.png and chart4_yearly_trend.png.

You : Which category has the worst reviews?
Bot : Category with the most negative reviews: Electronics.
Recommendation: Focus quality improvements here first.

You : Which platform has the most negative reviews?
Bot : Top 5 

In [8]:
# ══════════════════════════════════════════════════════════════
# SUMMARY REPORT
# ══════════════════════════════════════════════════════════════
print("\n📊  BUSINESS INTELLIGENCE SUMMARY REPORT")
print("=" * 62)
print(f"  Total Reviews Analysed  : {insights['total']:,}")
print(f"  Positive Reviews        : {insights['pos_pct']}%")
print(f"  Negative Reviews        : {insights['neg_pct']}%")
print(f"  Neutral  Reviews        : {insights['neu_pct']}%")
print(f"  Avg Sentiment Score     : {insights['avg_score']}  (-1 to +1)")
print(f"  Worst Category          : {insights['worst_cat']}")
print(f"  Worst Platform          : {insights['worst_platform']}")
print(f"  Best Month (sentiment)  : {insights['best_month']}")
print("\n  Top 5 Customer Complaints:")
for idx, c in enumerate(insights["top_complaints"], 1):
    print(f"    {idx}. {c}")
print("\n  Charts Saved:")
for chart in ["chart1_sentiment_pie.png", "chart2_category_bar.png",
              "chart3_monthly_trend.png", "chart4_yearly_trend.png"]:
    print(f"    {chart}")
print("=" * 62)






📊  BUSINESS INTELLIGENCE SUMMARY REPORT
  Total Reviews Analysed  : 40,000
  Positive Reviews        : 56.5%
  Negative Reviews        : 25.3%
  Neutral  Reviews        : 18.2%
  Avg Sentiment Score     : 0.192  (-1 to +1)
  Worst Category          : Electronics
  Worst Platform          : Amazon
  Best Month (sentiment)  : Jul

  Top 5 Customer Complaints:
    1. Product smells bad and looks nothing like described.
    2. Stopped working after a week. Awful.
    3. Average quality for the price.
    4. False advertising. Nothing like the photos.
    5. Never again. The worst purchase ever.

  Charts Saved:
    chart1_sentiment_pie.png
    chart2_category_bar.png
    chart3_monthly_trend.png
    chart4_yearly_trend.png


In [9]:
print("\n--- Interactive Mode (type 'exit' to quit) ---")
while True:
     user_input = input("You: ").strip()
     if user_input.lower() == "exit":
         print("Bot: Goodbye!")
         break
     print(f"Bot: {chatbot(user_input, insights)}\n")




--- Interactive Mode (type 'exit' to quit) ---
Bot: I didn't understand that. Type 'help' to see available questions.

Bot: Questions I can answer:
  1. What is overall customer sentiment?
  2. What are the top complaints?
  3. How has sentiment changed over time?
  4. Which category has the worst reviews?
  5. Which platform has the most negative reviews?
  6. What are your recommendations?
  7. How many total reviews were analysed?
  8. What is the average sentiment score?
  9. How many customers are satisfied?
  Type 'exit' to quit.

Bot: Best month for customer sentiment: Jul.
See chart3_monthly_trend.png and chart4_yearly_trend.png.

Bot: Goodbye!
